In [1]:
!pip uninstall -y crewai agentops
!pip uninstall -y numpy

!pip install "numpy<2.0"

Found existing installation: numpy 2.0.2
Uninstalling numpy-2.0.2:
  Successfully uninstalled numpy-2.0.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 77.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tobler 0.13.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.2 requires 

In [2]:
import numpy as np
print(np.__version__)

2.0.2


In [3]:
!pip uninstall -y crewai agentops

In [1]:
!pip install crewai==0.120.0 agentops

  Using cached crewai-0.120.0-py3-none-any.whl.metadata (33 kB)
  Using cached agentops-0.4.21-py3-none-any.whl.metadata (2.1 kB)
  Using cached appdirs-1.4.4-py2.py3-none-any.whl.metadata (9.0 kB)
  Using cached auth0_python-5.3.0-py3-none-any.whl.metadata (13 kB)
  Using cached chromadb-1.5.7-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (5.0 kB)
  Using cached instructor-1.15.1-py3-none-any.whl.metadata (12 kB)
  Using cached json_repair-0.59.2-py3-none-any.whl.metadata (18 kB)
  Using cached json5-0.14.0-py3-none-any.whl.metadata (36 kB)
  Using cached jsonref-1.1.0-py3-none-any.whl.metadata (2.7 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.8 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of instructor to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of instructor

In [2]:
!pip install -qU tavily-python

In [3]:
!pip install scrapegraph-py

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.3/49.3 kB 1.7 MB/s eta 0:00:00


In [4]:
from crewai import Agent,Task,Crew,Process,LLM
from crewai.tools import tool
import agentops
import os
from google.colab import userdata
from pydantic import BaseModel,Field
from typing import List
from tavily import TavilyClient
from scrapegraph_py import Client
import json
from crewai.knowledge.source.string_knowledge_source import StringKnowledgeSource

In [5]:
os.environ["Groq_API_Key"] = userdata.get('Groq_API_Key')
os.environ["AgentOps_API_Key"] = userdata.get('AgentOps_API_Key')

agentops.init(
    api_key=os.environ["AgentOps_API_Key"],
    skip_auto_end_session=True
)

In [7]:
os.environ["Tavily_API_Key"] = userdata.get('Tavily_API_Key')
os.environ["Scrapegraph_AI"] = userdata.get('Scrapegraph_AI')

#Tavily Client
search_client = TavilyClient(api_key=os.environ["Tavily_API_Key"])

#Scrapegraph AI client
scrape_client = Client(api_key=os.environ["Scrapegraph_AI"])

In [8]:
output_dir = './ai-agent-output'
os.makedirs(output_dir, exist_ok=True)

In [9]:
groq_key = userdata.get('Groq_API_Key')
os.environ["Gemini_KPI_Key"] = userdata.get('Gemini_KPI_Key')

In [22]:
basic_llm = LLM(
    model="groq/llama-3.3-70b-versatile",
    api_key=groq_key,
    temperature=0.0,          # مهم: يخلي الإخراج أكثر استقراراً وأقل tokens
    max_tokens=1000,          # حدد عشان ما يولدش نصوص طويلة زيادة
)

# SetUp Agent

## Agent A

In [23]:
no_keywords = 2

In [24]:
class SuggestedSearchQueries(BaseModel):
    queries: List[str] = Field(..., title = "suggested search queries to be passed to the search engine."
                                , min_length = 1, max_length = no_keywords)

class SingleSearchResul(BaseModel):
    title: str
    url: str = Field(...,title = "The Page URL")
    content:str
    score: float
    search_query: str

class AllSearchResults(BaseModel):
    results: List[SingleSearchResul]

In [25]:
search_queries_recommendation_agent = Agent(
    role = "Search Queries Recommendation Agent",
    goal = "/n".join(["To provide a list of search queries to be passed to the search engine",
                      "The queries must be varied and looking for specific items"]),
    backstory = "The agent is designed to help in looking for products by providing a list of suggested list queries",
    llm = basic_llm,
    verbose = True,
    max_rpm = 1,
    max_iter=3,
)

search_queries_recommendation_task = Task(
    description="\n".join([
        "Rankyx is looking to buy {product_name} at the best prices (value for a price strategy)",
        "The campany target any of these websites to buy from: {websites_list}",
        "The company wants to reach all available proucts on the internet to be compared later in another stage.",
        "The stores must sell the product in {country_name}",
        "Generate at maximum {no_keywords} queries.",
        "The search keywords must be in {language} language.",
        "Search keywords must contains specific brands, types or technologies. Avoid general keywords.",
        "The search query must reach an ecommerce webpage for product, and not a blog or listing page."
    ]),
    expected_output="A JSON object containing a list of suggested search queries.",
    output_json=SuggestedSearchQueries,
    output_file=os.path.join(output_dir, "step_1_suggested_search_queries.json"),
    agent=search_queries_recommendation_agent
)

## Agent B

In [26]:
@tool
def search_engine_tool(search_query: str) -> AllSearchResults:
  """Useful for search-based queries. Use this to find current information about any query related pages using a search engine"""
  return search_client.search(search_query)

search_engine_agent = Agent(
    role="Search Engine Agent",
    goal="To search for products based on the suggested search query",
    backstory="/n".join(["The agent is designed to help in looking for products by searching for products based on the suggested search queries."]),
    llm=basic_llm,
    verbose=True,
    tools=[search_engine_tool],
    max_rpm = 1,
    max_iter=3,
)

search_engine_task = Task(
    description="\n".join([
        "The task is to search for products based on the suggested search queries.",
        "You have to collect results from multiple search queries.",
        "Ignore any susbicious links or not an ecommerce single product website link.",
        "Ignore any search results with confidence score less than ({score_th}) .",
        "The search results will be used to compare prices of products from different websites.",
        "Pick the best 2 unique products",
    ]),
    expected_output="A JSON object containing the search results.",
    output_json=AllSearchResults,
    output_file=os.path.join(output_dir, "step_2_search_results.json"),
    agent=search_engine_agent
)

# Agent C

In [27]:
# url = "https://www.amazon.eg/-/en/Nescafe-Dolce-Gusto-Coffee-Machine/dp/B09CFH57N2"
# required_fields = ["product title","current price","old price"]

In [28]:
class ProductSpec(BaseModel):
    specification_name: str
    specification_value: str

class SingleExtractedProduct(BaseModel):
    page_url: str = Field(..., title="The original url of the product page")
    product_title: str = Field(..., title="The title of the product")
    product_image_url: str = Field(..., title="The url of the product image")
    product_url: str = Field(..., title="The url of the product")
    product_current_price: float = Field(..., title="The current price of the product")
    product_original_price: float = Field(title="The original price of the product before discount. Set to None if no discount", default=None)
    product_discount_percentage: float = Field(title="The discount percentage of the product. Set to None if no discount", default=None)

    product_specs: List[ProductSpec] = Field(..., title="The specifications of the product. Focus on the most important specs to compare.", min_items=1, max_items=5)

    agent_recommendation_rank: int = Field(..., title="The rank of the product to be considered in the final procurement report. (out of 5, Higher is Better) in the recommendation list ordering from the best to the worst")
    agent_recommendation_notes: List[str]  = Field(..., title="A set of notes why would you recommend or not recommend this product to the company, compared to other products.")


class AllExtractedProducts(BaseModel):
    products: List[SingleExtractedProduct]

/tmp/ipykernel_15383/1774330624.py:14: PydanticDeprecatedSince20: `min_items` is deprecated and will be removed, use `min_length` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  product_specs: List[ProductSpec] = Field(..., title="The specifications of the product. Focus on the most important specs to compare.", min_items=1, max_items=5)
/tmp/ipykernel_15383/1774330624.py:14: PydanticDeprecatedSince20: `max_items` is deprecated and will be removed, use `max_length` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  product_specs: List[ProductSpec] = Field(..., title="The specifications of the product. Focus on the most important specs to compare.", min_items=1, max_items=5)


In [29]:
@tool
def web_scraping_tool(page_url: str):
  """
    An AI Tool to help an agent to scrape a web page

    Example:
    web_scraping_tool(
        page_url="https://www.noon.com/egypt-en/15-bar-fully-automatic-espresso-machine-1-8-l-1500"
    )
    """
  schema_string = json.dumps(SingleExtractedProduct.model_json_schema())

  details = scrape_client.smartscraper(
      website_url=page_url,
      user_prompt=f"Extract ```json\n{schema_string}```\n from the web page."
  )
  return {"page_url":page_url,"details":details}


scraping_agent = Agent(
    role="Web scraping agent",
    goal="To extract details from any website",
    backstory="The agent is designed to help in looking for required values from any website url. These details will be used to decide which best product to buy.",
    llm=basic_llm,
    tools=[web_scraping_tool],
    verbose=True,
    max_rpm = 1,
    max_iter=3,
)

scraping_task = Task(
    description="\n".join([
        "The task is to extract product details from any ecommerce store page url.",
        "The task has to collect results from multiple pages urls.",
        "Collect the best {top_recommendations_no} products from the search results.",
    ]),
    expected_output="A JSON object containing products details",
    output_json=AllExtractedProducts,
    output_file=os.path.join(output_dir, "step_3_search_results.json"),
    agent=scraping_agent
)

# Agent D

In [30]:
procurement_report_author_agent = Agent(
    role="Procurement Report Author Agent",
    goal="To generate a professional HTML page for the procurement report",
    backstory="The agent is designed to assist in generating a professional HTML page for the procurement report after looking into a list of products.",
    llm=basic_llm,
    verbose=True,
    max_rpm = 1,
    max_iter=2,
)

procurement_report_author_task = Task(
    description="\n".join([
        "The task is to generate a professional HTML page for the procurement report.",
        "You have to use Bootstrap CSS framework for a better UI.",
        "Use the provided context about the company to make a specialized report.",
        "The report will include the search results and prices of products from different websites.",
        "The report should be structured with the following sections:",
        "1. Executive Summary: A brief overview of the procurement process and key findings.",
        "2. Introduction: An introduction to the purpose and scope of the report.",
        "3. Methodology: A description of the methods used to gather and compare prices.",
        "4. Findings: Detailed comparison of prices from different websites, including tables and charts.",
        "5. Analysis: An analysis of the findings, highlighting any significant trends or observations.",
        "6. Recommendations: Suggestions for procurement based on the analysis.",
        "7. Conclusion: A summary of the report and final thoughts.",
        "8. Appendices: Any additional information, such as raw data or supplementary materials.",
    ]),

    expected_output="A professional HTML page for the procurement report.",
    output_file=os.path.join(output_dir, "step_4_procurement_report.html"),
    agent=procurement_report_author_agent,
)

# Run AI Crew

In [31]:
# Create a knowledge source
about_company = "Rankyx is a company that provides AI solutions to help websites refine their search and recommendation systems."
company_context = StringKnowledgeSource(content=about_company)

rankyx_crew = Crew(
    agents= [
        search_queries_recommendation_agent,search_engine_agent,scraping_agent,procurement_report_author_agent
    ],
    # the sorting of the tasks is very important rather that agents
    tasks= [
        search_queries_recommendation_task,search_engine_task,scraping_task,procurement_report_author_task
    ],
    process = Process.sequential,
    knowledge_sources=[company_context],
)

In [32]:
#!uv add 'crewai[litellm]'

In [33]:
crew_results = rankyx_crew.kickoff(
    inputs = {
        "product_name": "coffee machine for the office",
        "websites_list": ["www.amazon.eg", "www.jumia.com.eg", "www.noon.com/egypt-en"],
        "country_name": "Egypt",
        "no_keywords": 2,
        "language": "English",
        "score_th": 0.10,
        "top_recommendations_no": 3
    }
)

# Agent: Search Queries Recommendation Agent
## Task: Rankyx is looking to buy coffee machine for the office at the best prices (value for a price strategy)
The campany target any of these websites to buy from: ['www.amazon.eg', 'www.jumia.com.eg', 'www.noon.com/egypt-en']
The company wants to reach all available proucts on the internet to be compared later in another stage.
The stores must sell the product in Egypt
Generate at maximum 2 queries.
The search keywords must be in English language.
Search keywords must contains specific brands, types or technologies. Avoid general keywords.
The search query must reach an ecommerce webpage for product, and not a blog or listing page.


# Agent: Search Queries Recommendation Agent
## Final Answer: 
{
  "queries": [
    "site:www.amazon.eg OR site:www.jumia.com.eg OR site:www.noon.com/egypt-en Nescafe coffee machine",
    "site:www.amazon.eg OR site:www.jumia.com.eg OR site:www.noon.com/egypt-en De'Longhi espresso machine"
  ]
}


# Agent: Se

🖇 AgentOps: ToolEvent() is deprecated and will be removed in v4 in the future. Automatically tracked in v4.




# Agent: Search Engine Agent
## Thought: Thought: I need to search for products based on the suggested search queries and collect results from multiple search queries, ignoring any suspicious links or non-ecommerce single product website links, and ignoring any search results with a confidence score less than 0.1.
## Using tool: search_engine_tool
## Tool Input: 
"{\"search_query\": \"site:www.amazon.eg OR site:www.jumia.com.eg OR site:www.noon.com/egypt-en Nescafe coffee machine\"}"
## Tool Output: 
{'query': 'OR OR Nescafe coffee machine', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://www.amazon.eg/-/en/Nescafe-Dolce-Gusto-Coffee-Machine/dp/B083XPQ9DZ', 'title': 'Nescafe Dolce Gusto Coffee Machine - White - EDG355.W1: Buy Online at Best Price in Egypt - Souq is now Amazon.eg', 'content': '# Product Summary: Nescafe Dolce Gusto Coffee Machine - White - EDG355.W1. ### About this Product. Nescafe Dolce Gusto Coffee Machine - White - EDG355.W1 

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)



Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Provider List: https://docs.litellm.ai/docs/providers


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Provider List: https://docs.litellm.ai/docs/providers


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Provider List: https://docs.litellm.ai/docs/providers


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Provider List: https://docs.litellm.ai/docs/providers


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Provider L

🖇 AgentOps: ErrorEvent() is deprecated and will be removed in v4 in the future. Automatically tracked in v4.


 

I encountered an error while trying to use the tool. This was the error: [402] Insufficient credits.
 Tool web_scraping_tool accepts these inputs: Tool Name: web_scraping_tool
Tool Arguments: {'page_url': {'description': None, 'type': 'str'}}
Tool Description: 
    An AI Tool to help an agent to scrape a web page

    Example:
    web_scraping_tool(
        page_url="https://www.noon.com/egypt-en/15-bar-fully-automatic-espresso-machine-1-8-l-1500"
    )
    



# Agent: Web scraping agent
## Thought: Thought: Now I have the filtered results for both search queries. I need to extract the product details from each of the top 3 unique product pages.
## Using tool: web_scraping_tool
## Tool Input: 
"{\"page_url\": \"https://www.amazon.eg/-/en/Nescafe-Dolce-Gusto-Coffee-Machine/dp/B083XPQ9DZ\"}"
## Tool Output: 

I encountered an error while trying to use the tool. This was the error: [402] Insufficient credits.
 Tool web_scraping_tool accepts these inputs: Tool Name: web_scraping_tool
T